In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
EDA para Capítulo 7 — (raw + secção sequencial derivada)
- Lê CSV raw (espera colunas: risk, event_id, time_to_tca)
- Gera figuras/tabelas compactas para o relatório
- Cria placeholders quando não há dados suficientes (nada "rebenta")
Saídas: reports/eda/* e reports/eda/seq/*
Uso:
  python eda_ch7.py --raw csv/train_data.csv --out reports/eda --threshold -6 --topk 20
"""
import os, json, math, argparse, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings("ignore")

# ------------------------- helpers -------------------------
def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def savefig(path, dpi=180):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()

def placeholder(path, msg="Sem dados suficientes para esta figura."):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    plt.figure(figsize=(6,3))
    plt.text(0.5,0.5,msg,ha="center",va="center",fontsize=11)
    plt.axis("off"); savefig(path)

def drop_leaks(df):
    # remove potenciais fugas de informação comuns neste domínio
    patterns = [r"max_risk", r"\bat_tca\b", r"peak", r"post_", r"future", r"after_tca",
                r"lastob", r"rcs_estimate", r"tca_flag", r"\blabel\b", r"\btarget\b"]
    leaks = [c for c in df.columns if any(re.search(p, c.lower()) for p in patterns)]
    if leaks:
        df = df.drop(columns=leaks)
    return df, leaks

def numeric_cols(df, exclude=("risk","y_bin","time_to_tca")):
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c not in exclude]

# ------------------------- main -------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", default="csv/train_data.csv")
    ap.add_argument("--out", default="reports/eda")
    ap.add_argument("--threshold", type=float, default=-6.0)
    ap.add_argument("--topk", type=int, default=20)  # para violinos/correlação
    ap.add_argument("--sample", type=int, default=150_000)  # limite de amostra p/ gráficos pesados
    args = ap.parse_args()

    OUT = ensure_dir(Path(args.out))
    SEQ_OUT = ensure_dir(OUT/"seq")
    THR = float(args.threshold)

    # ---------- carregar ----------
    df = pd.read_csv(args.raw)
    df = df.replace([np.inf,-np.inf], np.nan)
    # colunas mínimas
    for req in ["risk","event_id","time_to_tca"]:
        if req not in df.columns:
            raise SystemExit(f"[ERRO] Coluna obrigatória ausente no raw: {req}")

    # alvo binário (row level)
    df["y_bin"] = (df["risk"] >= THR).astype(int)
    # remover fugas
    df, leaks = drop_leaks(df)

    # ---------- tabelas rápidas ----------
    types = df.dtypes.value_counts().rename_axis("dtype").reset_index(name="count")
    types.to_csv(OUT/"dtypes_count.csv", index=False)

    miss = (df.isna().sum()
              .rename("n_missing")
              .to_frame()
              .reset_index().rename(columns={"index":"feature"})
            ).sort_values("n_missing", ascending=False)
    miss["pct_missing"] = 100*miss["n_missing"]/len(df)
    total_missing = int(miss["n_missing"].sum())
    miss["share_total_missing"] = 100*miss["n_missing"]/max(total_missing,1)
    miss.query("n_missing>0").head(30).to_csv(OUT/"missing_top30.csv", index=False)

    cls_counts = df["y_bin"].value_counts().rename_axis("class").reset_index(name="count")
    cls_counts["ratio"] = cls_counts["count"]/cls_counts["count"].sum()
    cls_counts.to_csv(OUT/"class_balance_row.csv", index=False)

    ev = df.sort_values(["event_id","time_to_tca"]).groupby("event_id")["risk"].max()
    ev_bin = (ev >= THR).astype(int)
    ev_counts = ev_bin.value_counts().rename_axis("class").reset_index(name="count")
    ev_counts["ratio"] = ev_counts["count"]/ev_counts["count"].sum()
    ev_counts.to_csv(OUT/"class_balance_event.csv", index=False)

    # ---------- gráficos básicos ----------
    # balance row
    try:
        plt.figure(figsize=(4.6,3.8))
        cls_counts.set_index("class")["ratio"].plot(kind="bar")
        plt.title("Row-level class ratio"); plt.ylabel("ratio"); plt.xticks(rotation=0)
        savefig(OUT/"balance_row.png")
    except Exception as e:
        placeholder(OUT/"balance_row.png", f"Falha a plotar: {e}")

    # balance event
    try:
        plt.figure(figsize=(4.6,3.8))
        ev_counts.set_index("class")["ratio"].plot(kind="bar")
        plt.title("Event-level class ratio (any_high_risk)"); plt.ylabel("ratio"); plt.xticks(rotation=0)
        savefig(OUT/"balance_event.png")
    except Exception as e:
        placeholder(OUT/"balance_event.png", f"Falha a plotar: {e}")

    # risk distribution + relação com tempo
    try:
        fig, ax = plt.subplots(1,2, figsize=(9.6,3.8))
        sns.histplot(df["risk"], bins=80, ax=ax[0])
        ax[0].axvline(THR, color="r", ls="--")
        ax[0].set_title("Distribuição de risk"); ax[0].set_xlabel("risk")
        ss = df if len(df)<=args.sample else df.sample(args.sample, random_state=42)
        sns.scatterplot(data=ss, x="time_to_tca", y="risk", hue="y_bin", s=8, alpha=0.4,
                        ax=ax[1], palette={0:"#557E55",1:"#914141"})
        ax[1].set_title("risk vs time_to_tca (amostra)")
        ax[1].legend(title="y_bin", loc="best", fontsize=8)
        savefig(OUT/"risk_overview.png")
    except Exception as e:
        placeholder(OUT/"risk_overview.png", f"Falha a plotar: {e}")

    # ---------- Top-K para violinos/correlação (ordenação por MI) ----------
    num_cols = numeric_cols(df)
    if len(num_cols)==0:
        placeholder(OUT/"violin_grid_topk.png", "Sem features numéricas (além de risk/time_to_tca).")
        placeholder(OUT/"corr/heatmap_top.png", "Sem features para correlação.")
        pd.DataFrame(columns=["feature","mutual_info"]).to_csv(OUT/"mutual_info_all.csv", index=False)
        pd.DataFrame(columns=["feature","pr_auc","roc_auc"]).to_csv(OUT/"univariate_auc.csv", index=False)
        top_feats = []
    else:
        samp = df if len(df)<=args.sample else df.sample(args.sample, random_state=42)
        X_mi = samp[num_cols].copy().fillna(samp[num_cols].median())
        y_mi = samp["y_bin"].values
        mi_vals = mutual_info_classif(X_mi, y_mi, random_state=42, discrete_features=False)
        mi_df = pd.DataFrame({"feature": num_cols, "mutual_info": mi_vals}).sort_values("mutual_info", ascending=False)
        mi_df.to_csv(OUT/"mutual_info_all.csv", index=False)

        # AP/ROC univariado
        rows = []
        for c in num_cols:
            s = samp[c].copy()
            if s.isna().all():
                rows.append({"feature":c, "roc_auc":np.nan, "pr_auc":np.nan}); continue
            s = s.fillna(s.median())
            x = (s - s.mean())/(s.std()+1e-9)
            try:
                roc = roc_auc_score(y_mi, x)
            except Exception:
                roc = np.nan
            pr = average_precision_score(y_mi, x)
            rows.append({"feature":c, "roc_auc":roc, "pr_auc":pr})
        uni_df = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
        uni_df.to_csv(OUT/"univariate_auc.csv", index=False)

        # violinos (grelha)
        top_feats = mi_df.head(args.topk)["feature"].tolist()
        pal = {"0": "#557E55", "1": "#914141"}
        n = len(top_feats)
        ncols = 4
        nrows = int(np.ceil(n/ncols)) if n else 1
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.6*nrows))
        axes = np.atleast_1d(axes).ravel()
        for ax, feat in zip(axes, top_feats):
            data = samp[[feat, "y_bin"]].dropna().copy()
            if data.empty: ax.axis("off"); continue
            data["y_bin"] = data["y_bin"].astype(str)
            sns.violinplot(data=data, x="y_bin", y=feat, inner="quartile", palette=pal, ax=ax)
            ax.set_xlabel("classe"); ax.set_title(feat, fontsize=9)
        for ax in axes[n:]: ax.axis("off")
        fig.suptitle("Distribuição por classe — Top features", y=1.01, fontsize=12)
        savefig(OUT/"violin_grid_topk.png")

        # heatmap correlação
        sub = samp[top_feats].copy().fillna(samp[top_feats].median()) if top_feats else pd.DataFrame()
        if not sub.empty:
            ensure_dir(OUT/"corr")
            plt.figure(figsize=(min(12, 0.6*len(top_feats)+2), min(10, 0.6*len(top_feats)+2)))
            sns.heatmap(sub.corr().round(2), cmap="coolwarm", center=0, annot=False, square=False)
            plt.title("Correlação entre Top features (MI)")
            savefig(OUT/"corr/heatmap_top.png")
        else:
            placeholder(OUT/"corr/heatmap_top.png", "Sem dados suficientes para heatmap.")

    # ---------- Pareto de missing nas principais (Top-K ou 30) ----------
    # usa Top-K se existir; senão calcula sobre todas as numéricas
    cols_for_pareto = top_feats if len(num_cols)>0 and len(top_feats)>0 else num_cols
    if len(cols_for_pareto)==0:
        placeholder(OUT/"missing_pareto_top30.png", "Sem colunas numéricas para Pareto de missing.")
        k80 = 0
    else:
        miss_counts = df[cols_for_pareto].isna().sum().sort_values(ascending=False)
        total = int(miss_counts.sum())
        if total == 0:
            placeholder(OUT/"missing_pareto_top30.png", "Sem missing nas colunas analisadas.")
            k80 = 0
        else:
            share = miss_counts/total*100.0
            cum = share.cumsum()
            cutoff = 80.0
            k = int(np.argmax(cum.values >= cutoff))
            plt.figure(figsize=(max(8, 0.32*len(share)+2), 5))
            plt.bar(range(len(share)), share.values)
            plt.ylabel("Quota do missing por coluna (%)")
            plt.xticks(range(len(share)), share.index, rotation=90)
            ax2 = plt.gca().twinx(); ax2.plot(range(len(cum)), cum.values, marker="o"); ax2.set_ylim(0,100)
            ax2.axhline(cutoff, ls="--"); plt.axvline(k, ls="--")
            plt.title("Missing (%) — Pareto nas principais colunas")
            plt.grid(alpha=0.2, ls=":")
            savefig(OUT/"missing_pareto_top30.png")
            k80 = k+1

    # ---------- Trajetórias de risco por evento (exemplos) ----------
    try:
        ev_ids = (df.groupby("event_id").size().sort_values(ascending=False).head(12).index.tolist())
        fig, axes = plt.subplots(3, 4, figsize=(12,8), sharex=True, sharey=True)
        axes = axes.ravel()
        for ax, ev_id in zip(axes, ev_ids):
            g = df[df["event_id"]==ev_id].sort_values("time_to_tca")
            ax.plot(g["time_to_tca"], g["risk"], lw=1.2)
            ax.axhline(THR, color="r", ls="--", lw=0.8)
            ax.set_title(f"event {ev_id}")
        for ax in axes[len(ev_ids):]:
            ax.axis("off")
        fig.suptitle("Evolução do risco por evento (exemplos)")
        savefig(OUT/"risk_trajectories_examples.png")
    except Exception as e:
        placeholder(OUT/"risk_trajectories_examples.png", f"Falha a plotar: {e}")

    # ================= [SEQ] EDA derivado (sem usar rótulos extra) =================
    # 1) comprimento por evento (hist)
    try:
        seq_len_series = df.groupby("event_id").size()
        if len(seq_len_series)>0:
            rec_seq_len = int(np.clip(np.percentile(seq_len_series, 90), 3, 200))
            plt.figure(figsize=(6.5,3.8))
            seq_len_series.plot(kind="hist", bins=30)
            plt.axvline(rec_seq_len, ls="--", label=f"p90={rec_seq_len}")
            plt.legend()
            plt.title(f"Comprimento por evento — sugestão seq_len (p90) = {rec_seq_len}")
            plt.xlabel("# observações por event_id")
            savefig(OUT/"seq/len_hist.png")
        else:
            placeholder(OUT/"seq/len_hist.png", "Sem eventos para estimar comprimentos.")
    except Exception as e:
        placeholder(OUT/"seq/len_hist.png", f"Falha a plotar: {e}")

    # 2) cobertura por bins de tempo_to_tca (quantis)
    try:
        tt = df["time_to_tca"].dropna().values
        if len(tt) < 5 or np.nanstd(tt)==0:
            placeholder(OUT/"seq/coverage_heatmap.png", "time_to_tca insuficiente para bins temporais.")
            pd.DataFrame().to_csv(OUT/"seq/coverage_bins.csv", index=False)
        else:
            SEQ_BINS = 10
            edges = np.quantile(tt, np.linspace(0,1,SEQ_BINS+1))
            bin_ids = np.digitize(df["time_to_tca"], edges[1:-1], right=True)
            bin_labels = [f"[{edges[i]:.2f},{edges[i+1]:.2f})" for i in range(SEQ_BINS)]
            feats = numeric_cols(df)[:min(30, len(numeric_cols(df)))]
            cov = []
            for f in feats:
                cov.append(df.groupby(bin_ids, dropna=False)[f].apply(lambda s: float(s.notna().mean())))
            coverage = pd.DataFrame(cov, index=feats).T
            coverage.index = bin_labels
            coverage.to_csv(OUT/"seq/coverage_bins.csv", index=False)
            plt.figure(figsize=(10, 0.35*len(feats)+2))
            sns.heatmap(coverage.T, cmap="viridis", vmin=0, vmax=1, cbar_kws={"label":"proporção não-nula"})
            plt.title("Cobertura (% não-missing) por bin de tempo e feature")
            savefig(OUT/"seq/coverage_heatmap.png")
    except Exception as e:
        placeholder(OUT/"seq/coverage_heatmap.png", f"Falha a plotar: {e}")

    # ---------- sumário ----------
    summary = {
        "n_rows": int(len(df)),
        "n_cols": int(df.shape[1]),
        "n_numeric": int(sum(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)),
        "n_categorical": int(sum(df[c].dtype=="object" for c in df.columns)),
        "row_pos_ratio": float(cls_counts.loc[cls_counts["class"]==1,"ratio"].values[0]) if (cls_counts["class"]==1).any() else float("nan"),
        "event_pos_ratio": float(ev_counts.loc[ev_counts["class"]==1,"ratio"].values[0]) if (ev_counts["class"]==1).any() else float("nan"),
        "threshold": THR,
        "leaks_removed": leaks,
        "topk_violin": int(args.topk),
        "pareto_80pct_cols": int(k80) if 'k80' in locals() else None,
        "artifacts": {
            "figs": [
                "risk_overview.png","balance_row.png","balance_event.png",
                "violin_grid_topk.png","corr/heatmap_top.png","missing_pareto_top30.png",
                "risk_trajectories_examples.png",
                "seq/len_hist.png","seq/coverage_heatmap.png"
            ],
            "tables": [
                "dtypes_count.csv","missing_top30.csv","class_balance_row.csv","class_balance_event.csv",
                "mutual_info_all.csv","univariate_auc.csv","seq/coverage_bins.csv"
            ]
        }
    }
    with open(OUT/"eda_summary.json","w") as f:
        json.dump(summary, f, indent=2)

    print("[OK] EDA (Cap.7) concluído →", OUT.resolve())
    print("Figuras:", *summary["artifacts"]["figs"], sep="\n - ")
    print("Tabelas:", *summary["artifacts"]["tables"], sep="\n - ")

if __name__ == "__main__":
    main()


Output dir: C:\Users\andal\OneDrive\Ambiente de Trabalho\cenas_satelites\reports\eda
shape: (162634, 103)


,event_id,time_to_tca,mission_id,risk,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
0,0,1.566798,5,-10.204955,-7.834756,8.602101,14923.0,13792.0,453.8,5976.6,...,0.147350,58.272095,0.004092,0.165044,0.002987,0.386462,89.0,83.0,42.0,11.0
1,0,1.207494,5,-10.355758,-7.848937,8.956374,14544.0,13792.0,474.3,5821.2,...,0.059672,57.966413,0.003753,0.164383,0.002933,0.386393,89.0,83.0,42.0,11.0
2,0,0.952193,5,-10.345631,-7.847406,8.932195,14475.0,13792.0,474.6,5796.2,...,0.039258,57.907599,0.003576,0.164352,0.002967,0.386381,89.0,83.0,42.0,11.0


[leakage] Removidas 8 colunas: ['max_risk_estimate', 'max_risk_scaling', 't_time_lastob_start', 't_time_lastob_end', 't_rcs_estimate', 'c_time_lastob_start', 'c_time_lastob_end', 'c_rcs_estimate'] 


,dtype,count
0,float64,90
1,int64,4
2,object,1
3,int32,1


,feature,n_missing,pct_missing,share_total_missing
61,c_ctdot_rdot,9241,5.682084,3.034961
88,c_sigma_tdot,9241,5.682084,3.034961
62,c_cndot_r,9241,5.682084,3.034961
66,c_cndot_tdot,9241,5.682084,3.034961
64,c_cndot_n,9241,5.682084,3.034961
63,c_cndot_t,9241,5.682084,3.034961
86,c_sigma_rdot,9241,5.682084,3.034961
55,c_crdot_r,9241,5.682084,3.034961
65,c_cndot_rdot,9241,5.682084,3.034961
56,c_crdot_t,9241,5.682084,3.034961


...

,class,count,ratio
0,0,11179,0.849856
1,1,1975,0.150144


numéricas: 93 | categóricas: 1


,feature,mutual_info
72,azimuth,0.154821
49,c_j2k_inc,0.152796
10,relative_velocity_n,0.139759
9,relative_velocity_t,0.138037
47,c_j2k_sma,0.130192
0,event_id,0.129880
22,t_j2k_inc,0.129065
70,c_h_per,0.127092
69,c_h_apo,0.121284
20,t_j2k_sma,0.119485


,feature,roc_auc,pr_auc
1,time_to_tca,0.677282,0.104099
20,t_j2k_sma,0.600839,0.098762
22,t_j2k_inc,0.631568,0.098596
67,t_h_apo,0.600340,0.098207
68,t_h_per,0.601006,0.097662
70,c_h_per,0.592386,0.093053
65,t_span,0.587446,0.089844
47,c_j2k_sma,0.599586,0.084273
69,c_h_apo,0.595688,0.081372
24,t_cn_r,0.555720,0.080968


Grelha de violinos -> reports\eda\violin_grid_topk.png
